In [1]:
1+1

2

In [1]:
import polars as pl

In [2]:
import seaborn as sns

In [6]:
INPUT_FILE = "politician_trades_data/congress_trades_full.parquet"

In [13]:
df = pl.read_parquet(INPUT_FILE)

In [14]:
df_agg = df['Ticker'].unique()
df_agg.to_csv('unique_tickers.csv',separator=';',decimal_comma=True)

AttributeError: 'Series' object has no attribute 'to_csv'

In [36]:
df.columns

['Ticker',
 'TradeDate',
 'Traded',
 'FiledDate',
 'Filed',
 'Name',
 'Transaction',
 'Trade_Size_USD',
 'Party',
 'Chamber',
 'Company',
 'Description',
 'trade_price',
 'filed_price',
 'traded_to_filed_pct',
 'return_1m_pct',
 'delta_1m_usd',
 'return_3m_pct',
 'delta_3m_usd',
 'return_6m_pct',
 'delta_6m_usd',
 'return_9m_pct',
 'delta_9m_usd',
 'return_12m_pct',
 'delta_12m_usd']

In [45]:
import yfinance as yf
from dateutil.relativedelta import relativedelta
from datetime import datetime, date

ticker = 'ENOB'
target_start = date(2022, 7, 18)
target_end = date(2023, 12, 12)

In [46]:
data = yf.download(ticker, start=target_start, end=target_end+relativedelta(days=1), multi_level_index=False, progress=False)
print(f'data', data)

$ENOB: possibly delisted; no timezone found

1 Failed download:
['ENOB']: possibly delisted; no timezone found


data Empty DataFrame
Columns: [Adj Close, Close, High, Low, Open, Volume]
Index: []


In [43]:
#ACXM,2018-04-02,2019-05-08

import yfinance as yf
from contextlib import redirect_stdout, redirect_stderr
import io
from datetime import date
from dateutil.relativedelta import relativedelta

ticker = "ACXM"  # your delisted ticker
target_start = date(2018, 4, 2)
target_end = date(2019, 5, 8)

# Create a string buffer to capture stdout/stderr
f = io.StringIO()

# Redirect stdout and stderr
with redirect_stdout(f), redirect_stderr(f):
    data = yf.download(
        ticker,
        start=target_start,
        end=target_end + relativedelta(days=1),
        progress=False,
        multi_level_index=False
    )

# Read everything printed by yfinance
yfinance_output = f.getvalue()

# Check if DataFrame is empty and capture message
if data.empty:
    warning_message = yfinance_output.strip()
else:
    warning_message = None

# Example usage
print("Captured message:", warning_message)
print("Data shape:", data.shape)


Captured message: /Users/ascaravelli/Documents/GitHub/Congress_stock_analysis/venv/lib/python3.14/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
$ACXM: possibly delisted; no price data found  (1d 2018-04-02 -> 2019-05-09)

1 Failed download:
['ACXM']: possibly delisted; no price data found  (1d 2018-04-02 -> 2019-05-09)
Data shape: (0, 6)


In [44]:
warning_message.replace("""/Users/ascaravelli/Documents/GitHub/Congress_stock_analysis/venv/lib/python3.14/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()""",'')

"\n$ACXM: possibly delisted; no price data found  (1d 2018-04-02 -> 2019-05-09)\n\n1 Failed download:\n['ACXM']: possibly delisted; no price data found  (1d 2018-04-02 -> 2019-05-09)"

In [29]:
import requests
import re

def get_yahoo_cookie_and_crumb():
    session = requests.Session()
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    r = session.get(
        "https://finance.yahoo.com/quote/SPY",
        headers=headers,
        timeout=10
    )
    r.raise_for_status()

    # Extract crumb
    match = re.search(r'"CrumbStore":\{"crumb":"(.*?)"\}', r.text)
    if not match:
        raise RuntimeError("Could not find Yahoo crumb")

    crumb = match.group(1).encode("ascii").decode("unicode_escape")
    return session, crumb


In [30]:
from datetime import datetime

def download_yahoo_history(symbol, start, end):
    session, crumb = get_yahoo_cookie_and_crumb()

    period1 = int(datetime(*start).timestamp())
    period2 = int(datetime(*end).timestamp())

    url = f"https://query1.finance.yahoo.com/v7/finance/download/{symbol}"

    params = {
        "period1": period1,
        "period2": period2,
        "interval": "1d",
        "events": "history",
        "includeAdjustedClose": "true",
        "crumb": crumb
    }

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "text/csv"
    }

    r = session.get(url, params=params, headers=headers, timeout=10)
    r.raise_for_status()
    return r.text


In [31]:
csv = download_yahoo_history(
    "AAIC-PB",
    (2015, 1, 8),
    (2021, 4, 15)
)


RuntimeError: Could not find Yahoo crumb

In [58]:
import requests
import json

API_KEY = "XQfuCF7MJSZWwgFxZF8pbzx6NfVfEdmY45Oebjqp"
#url = "https://api.congress.gov/v3/member/C001098"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}
#url = "https://api.congress.gov/v3/committee"
url = "https://api.congress.gov/committee/senate-finance/ssfi?congress=117"
r = requests.get(url, params={"api_key": API_KEY}, headers=HEADERS)
print(r)
print(json.dumps(r.json(), indent=2))

<Response [404]>


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [59]:
import yfinance as yf

# A rule-based map of Industry -> Committee
JURISDICTION_MAP = {
    # --- Defense & Aerospace ---
    "Aerospace & Defense": ["House Armed Services", "Senate Armed Services"],
    
    # --- Finance ---
    "Banks": ["House Financial Services", "Senate Banking, Housing, and Urban Affairs"],
    "Capital Markets": ["House Financial Services", "Senate Banking"],
    "Insurance": ["House Financial Services", "Senate Banking"],
    
    # --- Healthcare ---
    "Drug Manufacturers": ["House Energy & Commerce", "Senate Health, Education, Labor & Pensions (HELP)"],
    "Biotechnology": ["House Energy & Commerce", "Senate HELP"],
    "Healthcare Plans": ["House Energy & Commerce", "Senate Finance"],
    
    # --- Energy ---
    "Oil & Gas": ["House Energy & Commerce", "Senate Energy & Natural Resources"],
    "Utilities": ["House Energy & Commerce", "Senate Energy & Natural Resources"],
    
    # --- Tech ---
    "Software": ["House Judiciary", "Senate Commerce, Science, & Transportation"],
    "Semiconductors": ["House Science, Space, & Tech", "Senate Commerce"],
    "Internet Content": ["House Energy & Commerce", "Senate Judiciary"], # Social media
    
    # --- Transport ---
    "Airlines": ["House Transportation & Infrastructure", "Senate Commerce"],
    "Railroads": ["House Transportation & Infrastructure", "Senate Commerce"],
}

def get_stock_committees(ticker):
    stock = yf.Ticker(ticker)
    try:
        info = stock.info
        industry = info.get('industry', 'Unknown')
        sector = info.get('sector', 'Unknown')
        
        print(f"--- Analysis for {ticker.upper()} ---")
        print(f"Sector:   {sector}")
        print(f"Industry: {industry}")
        
        # 1. Direct Industry Match
        committees = []
        for key in JURISDICTION_MAP:
            if key in industry:
                committees.extend(JURISDICTION_MAP[key])
        
        # 2. Fallback: Sector Match (Broad)
        if not committees:
            if sector == "Financial Services":
                committees.append("House Financial Services")
            elif sector == "Healthcare":
                committees.append("House Energy & Commerce")
            elif sector == "Energy":
                committees.append("Senate Energy & Natural Resources")
            elif sector == "Technology":
                committees.append("Senate Commerce, Science, & Transportation")

        if committees:
            print(f">> Primary Committees: {', '.join(set(committees))}")
        else:
            print(">> No direct committee match found (check manual lookup).")
            
    except Exception as e:
        print(f"Error looking up {ticker}: {e}")

# Try it out
get_stock_committees("LMT")  # Lockheed
get_stock_committees("JPM")  # JP Morgan


--- Analysis for LMT ---
Sector:   Industrials
Industry: Aerospace & Defense
>> Primary Committees: House Armed Services, Senate Armed Services
--- Analysis for JPM ---
Sector:   Financial Services
Industry: Banks - Diversified
>> Primary Committees: Senate Banking, Housing, and Urban Affairs, House Financial Services


/Users/ascaravelli/Documents/GitHub/Congress_stock_analysis/venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/Users/ascaravelli/Documents/GitHub/Congress_stock_analysis/venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:704: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")


In [60]:
from yahooquery import Ticker

# List of tickers from your congressional dataset
symbols = ['AAPL', 'MSFT', 'PFE', 'XOM']

# Fetch 'summaryProfile' module (contains sector/industry)
tickers = Ticker(symbols, asynchronous=True)
data = tickers.get_modules('summaryProfile quoteType')

# The result is a dictionary you can easily convert to a DataFrame
# print(data['AAPL']['summaryProfile']['sector'])

In [61]:
data

{'AAPL': {'quoteType': {'exchange': 'NMS',
   'quoteType': 'EQUITY',
   'symbol': 'AAPL',
   'underlyingSymbol': 'AAPL',
   'shortName': 'Apple Inc.',
   'longName': 'Apple Inc.',
   'firstTradeDateEpochUtc': '1980-12-12 22:00:00',
   'timeZoneFullName': 'America/New_York',
   'timeZoneShortName': 'EST',
   'uuid': '8b10e4ae-9eeb-3684-921a-9ab27e4d87aa',
   'messageBoardId': 'finmb_24937',
   'gmtOffSetMilliseconds': -18000000,
   'maxAge': 1},
  'summaryProfile': {'address1': 'One Apple Park Way',
   'city': 'Cupertino',
   'state': 'CA',
   'zip': '95014',
   'country': 'United States',
   'phone': '(408) 996-1010',
   'website': 'https://www.apple.com',
   'industry': 'Consumer Electronics',
   'industryKey': 'consumer-electronics',
   'industryDisp': 'Consumer Electronics',
   'sector': 'Technology',
   'sectorKey': 'technology',
   'sectorDisp': 'Technology',
   'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearable

In [68]:
import json
from yahooquery import Ticker

# 1. Simulate your JSON structure with just 2 examples
# (In reality, you'd load this from your file, but this is safer for testing)
mock_data = {
    "tickers": {
        "AAPL": {
            "min_date": "1980-12-12",
            "max_date": "2023-10-21"
        },
        "MSFT": {
            "min_date": "1986-03-13",
            "max_date": "2023-10-21"
        }
    }
}

# Extract tickers from the mock data
test_tickers = list(mock_data.get("tickers", {}).keys())
print(f"Testing with tickers: {test_tickers}")

# 2. Fetch Data
# asynchronous=True is good practice even for small batches
t = Ticker(test_tickers, asynchronous=True)
response = t.get_modules('summaryProfile')

# 3. Process and Print (No Saving)
print("\n--- RESULTS ---\n")

for symbol in test_tickers:
    print(f"Processing: {symbol}")
    
    # Check if data exists and is a dictionary (not an error string)
    if symbol in response and isinstance(response[symbol], dict):
        profile = response[symbol]
        
        if profile:
            # Simulate adding it to the structure
            mock_data['tickers'][symbol]['summaryProfile'] = profile
            
            # PRINT THE RESULT
            print(f"  -> Success! Found Sector: {profile.get('sector', 'N/A')}")
            print(f"  -> Industry: {profile.get('industry', 'N/A')}")
            print(f"  -> Full Data Snippet: {str(profile)[:100]}...") # Print first 100 chars
        else:
            print("  -> 'summaryProfile' key missing in response.")
    else:
        print(f"  -> Failed to fetch data. API Response: {response.get(symbol)}")

    print("-" * 30)

print("\n--- FINAL JSON STRUCTURE (Preview) ---")
# Print a pretty version of one entry to see how it looks nested
print(json.dumps(mock_data['tickers']['AAPL'], indent=4))


Testing with tickers: ['AAPL', 'MSFT']

--- RESULTS ---

Processing: AAPL
  -> Success! Found Sector: Technology
  -> Industry: Consumer Electronics
  -> Full Data Snippet: {'address1': 'One Apple Park Way', 'city': 'Cupertino', 'state': 'CA', 'zip': '95014', 'country': 'U...
------------------------------
Processing: MSFT
  -> Success! Found Sector: Technology
  -> Industry: Software - Infrastructure
  -> Full Data Snippet: {'address1': 'One Microsoft Way', 'city': 'Redmond', 'state': 'WA', 'zip': '98052-6399', 'country': ...
------------------------------

--- FINAL JSON STRUCTURE (Preview) ---
{
    "min_date": "1980-12-12",
    "max_date": "2023-10-21",
    "summaryProfile": {
        "address1": "One Apple Park Way",
        "city": "Cupertino",
        "state": "CA",
        "zip": "95014",
        "country": "United States",
        "phone": "(408) 996-1010",
        "website": "https://www.apple.com",
        "industry": "Consumer Electronics",
        "industryKey": "consumer-

In [67]:
response[symbol]

{'address1': 'One Microsoft Way',
 'city': 'Redmond',
 'state': 'WA',
 'zip': '98052-6399',
 'country': 'United States',
 'phone': '425 882 8080',
 'website': 'https://www.microsoft.com',
 'industry': 'Software - Infrastructure',
 'industryKey': 'software-infrastructure',
 'industryDisp': 'Software - Infrastructure',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': "Microsoft Corporation develops and supports software, services, devices, and solutions worldwide. The company's Productivity and Business Processes segment offers Microsoft 365 Commercial, Enterprise Mobility + Security, Windows Commercial, Power BI, Exchange, SharePoint, Microsoft Teams, Security and Compliance, and Copilot; Microsoft 365 Commercial products, such as Windows Commercial on-premises and Office licensed services; Microsoft 365 Consumer products and cloud services, such as Microsoft 365 Consumer subscriptions, Office licensed on-premises, and other consu

In [70]:
import json

# 1. Load the file
file_path = 'stock_data/metadata_enriched.json' 

print(f"Reading {file_path}...")

try:
    with open(file_path, 'r') as f:
        data = json.load(f)

    tickers_data = data.get('tickers', {})
    total_tickers = len(tickers_data)
    
    # Counters and Sets
    missing_sector_count = 0
    missing_industry_count = 0
    
    unique_sectors = set()
    unique_industries = set()

    # 2. Loop through all tickers
    for symbol, info in tickers_data.items():
        
        # Navigate safely: info -> summaryProfile
        profile = info.get('summaryProfile', {})
        if profile is None: 
            profile = {}
            
        # Extract Data
        sector = profile.get('sector')
        industry = profile.get('industry')

        # 3. Analyze Sector
        if sector:
            unique_sectors.add(sector)
        else:
            missing_sector_count += 1
            
        # 4. Analyze Industry
        if industry:
            unique_industries.add(industry)
        else:
            missing_industry_count += 1

    # 5. Print Results
    print("\n" + "="*40)
    print("          ANALYSIS REPORT       ")
    print("="*40)
    print(f"Total Tickers: {total_tickers}")
    print("-" * 40)
    print(f"Missing Sector:   {missing_sector_count} ({(missing_sector_count/total_tickers)*100:.1f}%)")
    print(f"Missing Industry: {missing_industry_count} ({(missing_industry_count/total_tickers)*100:.1f}%)")
    print("-" * 40)
    
    # Print Sectors
    sorted_sectors = sorted(list(unique_sectors))
    print(f"\nDistinct SECTORS ({len(sorted_sectors)}):")
    for s in sorted_sectors:
        print(f" - {s}")

    # Print Industries
    sorted_industries = sorted(list(unique_industries))
    print(f"\nDistinct INDUSTRIES ({len(sorted_industries)}):")
    for i in sorted_industries:
        print(f" - {i}")

except FileNotFoundError:
    print(f"Error: Could not find file '{file_path}'.")
except json.JSONDecodeError:
    print(f"Error: '{file_path}' is not a valid JSON file.")


Reading stock_data/metadata_enriched.json...

          ANALYSIS REPORT       
Total Tickers: 2630
----------------------------------------
Missing Sector:   337 (12.8%)
Missing Industry: 337 (12.8%)
----------------------------------------

Distinct SECTORS (11):
 - Basic Materials
 - Communication Services
 - Consumer Cyclical
 - Consumer Defensive
 - Energy
 - Financial Services
 - Healthcare
 - Industrials
 - Real Estate
 - Technology
 - Utilities

Distinct INDUSTRIES (141):
 - Advertising Agencies
 - Aerospace & Defense
 - Agricultural Inputs
 - Airlines
 - Airports & Air Services
 - Aluminum
 - Apparel Manufacturing
 - Apparel Retail
 - Asset Management
 - Auto & Truck Dealerships
 - Auto Manufacturers
 - Auto Parts
 - Banks - Diversified
 - Banks - Regional
 - Beverages - Brewers
 - Beverages - Non-Alcoholic
 - Beverages - Wineries & Distilleries
 - Biotechnology
 - Broadcasting
 - Building Materials
 - Building Products & Equipment
 - Business Equipment & Supplies
 - Capital Ma

In [73]:
import requests

url = "https://lda.senate.gov/api/v1/filings/"
params = {
    "client_name": "Pfizer Inc",
    "filing_year": "2024",
    "filing_period": "first_quarter"
}

r = requests.get(url, params=params)
data = r.json()

# Look at the first report found
if data['results']:
    filing = data['results'][0]
    print(f"Lobbyist: {filing['registrant']['name']}")
    print(f"Specific Issues: {filing['general_issue_code_display']}")
    print(f"Government Bodies Lobbied: {filing['government_entities']}")


Lobbyist: ALTRIUS GROUP, LLC


KeyError: 'general_issue_code_display'

In [74]:
data

{'count': 13,
 'next': None,
 'previous': None,
 'results': [{'url': 'https://lda.senate.gov/api/v1/filings/c90d081f-5a7e-43b9-9233-c596d973e46e/',
   'filing_uuid': 'c90d081f-5a7e-43b9-9233-c596d973e46e',
   'filing_type': 'Q1',
   'filing_type_display': '1st Quarter - Report',
   'filing_year': 2024,
   'filing_period': 'first_quarter',
   'filing_period_display': '1st Quarter (Jan 1 - Mar 31)',
   'filing_document_url': 'https://lda.senate.gov/filings/public/filing/c90d081f-5a7e-43b9-9233-c596d973e46e/print/',
   'filing_document_content_type': 'text/html',
   'income': '20000.00',
   'expenses': None,
   'expenses_method': None,
   'expenses_method_display': None,
   'posted_by_name': 'WILLIAM J. MORLEY',
   'dt_posted': '2024-04-17T13:59:06-04:00',
   'termination_date': None,
   'registrant_country': 'United States of America',
   'registrant_ppb_country': None,
   'registrant_address_1': '2000 Pennsylvania Ave, NW',
   'registrant_address_2': 'Suite 1030',
   'registrant_differe

In [79]:
import json
import yaml

# CONFIG
STOCK_FILE = 'stock_data/metadata_enriched.json'
MAP_FILE = 'commette_industry_map.yaml'

def load_data():
    with open(STOCK_FILE, 'r') as f:
        stock_data = json.load(f)
    with open(MAP_FILE, 'r') as f:
        rules_data = yaml.safe_load(f)
    return stock_data.get('tickers', {}), rules_data.get('committees', {})

def check_coverage():
    tickers, committees = load_data()
    
    unassigned_no_data = [] # Stocks that have NO sector/industry info (ETFs, errors)
    unassigned_orphan = []  # Stocks that HAVE data, but didn't match any rule (Logic gaps)
    
    print(f"Scanning {len(tickers)} stocks...")
    
    for symbol, info in tickers.items():
        is_mapped = False
        
        # Get data
        profile = info.get('summaryProfile', {}) or {}
        stock_sector = profile.get('sector')
        stock_industry = profile.get('industry')
        
        # Case 1: No Data (Skip logic, just log it)
        if not stock_sector and not stock_industry:
            unassigned_no_data.append(symbol)
            continue
            
        # Case 2: Has Data -> Check for Matches
        for comm_code, comm_rules in committees.items():
            # Check Industry
            if stock_industry and stock_industry in comm_rules.get('industries', {}):
                is_mapped = True
                break
            # Check Sector
            if stock_sector and stock_sector in comm_rules.get('sectors', {}):
                is_mapped = True
                break
        
        if not is_mapped:
            # Record the orphan stock and its sector so you can fix your map
            unassigned_orphan.append(f"{symbol} [{stock_sector} | {stock_industry}]")

    # --- REPORT ---
    print("\n" + "="*50)
    print("           UNASSIGNED STOCKS REPORT")
    print("="*50)
    
    print(f"\n1. MISSING DATA ({len(unassigned_no_data)} stocks)")
    print("   (Likely ETFs, Funds, or Delisted tickers)")
    print(f"   Examples: {unassigned_no_data[:10]} ...")
    
    print(f"\n2. ORPHAN STOCKS ({len(unassigned_orphan)} stocks)")
    print("   (Stocks with valid sectors that your YAML map missed)")
    print("-" * 50)
    
    if unassigned_orphan:
        # Sort them to help you find patterns
        unassigned_orphan.sort()
        for item in unassigned_orphan:
            print(f"   [X] {item}")
    else:
        print("   Good news! Every stock with data was successfully mapped.")
        
    print("="*50)

if __name__ == "__main__":
    check_coverage()


Scanning 2630 stocks...

           UNASSIGNED STOCKS REPORT

1. MISSING DATA (337 stocks)
   (Likely ETFs, Funds, or Delisted tickers)
   Examples: ['AAXJ', 'ABNFX', 'ABQCX', 'ACEIX', 'ACSDX', 'AET', 'AEX.MU', 'AGG', 'AGZ', 'AHGP'] ...

2. ORPHAN STOCKS (0 stocks)
   (Stocks with valid sectors that your YAML map missed)
--------------------------------------------------
   Good news! Every stock with data was successfully mapped.


In [95]:
import yaml
import json
import os
import glob
import sys
from datetime import datetime

# --- Configuration ---
# Ensure these match your folder structure
METADATA_FILE = 'stock_data/metadata_enriched.json'
COMMITTEE_DIR = 'committee_history'
MAP_FILE = 'commette_industry_map.yaml'

def parse_file_date(filename):
    try:
        base_name = os.path.basename(filename)
        # Assumes format: 2012-11-09T20-29-07Z_...
        timestamp_str = base_name.split('_')[0]
        return datetime.strptime(timestamp_str, "%Y-%m-%dT%H-%M-%SZ")
    except (ValueError, IndexError):
        return None

def get_closest_past_file(target_date, committee_dir):
    all_files = glob.glob(os.path.join(committee_dir, "*.yaml"))
    valid_files = []
    
    for f in all_files:
        d = parse_file_date(f)
        if d:
            valid_files.append((d, f))
    
    # Filter: File date must be strictly LESS than trade date
    past_files = [x for x in valid_files if x[0] < target_date]
    
    if not past_files:
        return None, None
        
    # Return the file with the largest date (closest to trade)
    return max(past_files, key=lambda x: x[0])

def test_transaction(ticker, bioguide_id, date_str):
    print(f"\n{'='*60}")
    print(f"TESTING TRANSACTION: {ticker} | {bioguide_id} | {date_str}")
    print(f"{'='*60}")
    
    try:
        trade_date = datetime.strptime(date_str, "%Y-%m-%d")
    except ValueError:
        print("❌ Error: Date format must be YYYY-MM-DD")
        return

    # --- Step 1: Ticker Metadata ---
    print(f"\n[1] Looking up Metadata for {ticker}...")
    try:
        with open(METADATA_FILE, 'r') as f:
            meta = json.load(f)
        
        ticker_data = meta.get('tickers', {}).get(ticker, {}).get('summaryProfile', {})
        industry = ticker_data.get('industry')
        sector = ticker_data.get('sector')
        
        if industry or sector:
            print(f"   ✅ Found Industry: {industry}")
            print(f"   ✅ Found Sector:   {sector}")
        else:
            print(f"   ⚠️  Ticker found, but no industry/sector data.")
            
    except Exception as e:
        print(f"   ❌ Error loading metadata: {e}")
        return

    # --- Step 2: Committee File Selection ---
    print(f"\n[2] Locating Committee History File...")
    file_date, file_path = get_closest_past_file(trade_date, COMMITTEE_DIR)
    
    if not file_path:
        print("   ⚠️  No committee file found prior to this trade date.")
        committees = []
    else:
        print(f"   ✅ Selected File: {os.path.basename(file_path)}")
        print(f"   📅 File Date:     {file_date}")
        
        # --- Step 3: Extract Committees ---
        print(f"\n[3] Finding Committees for {bioguide_id} in this file...")
        committees = []
        try:
            with open(file_path, 'r') as f:
                comm_data = yaml.safe_load(f)
                
            if comm_data:
                for code, members in comm_data.items():
                    if members:
                        for m in members:
                            if isinstance(m, dict) and m.get('bioguide') == bioguide_id:
                                committees.append(code)
            
            if committees:
                print(f"   ✅ Member found in: {committees}")
            else:
                print(f"   ⚠️  Member {bioguide_id} not found in any committee in this file.")
            
        except Exception as e:
            print(f"   ❌ Error reading committee file: {e}")
            return

    # --- Step 4: Map Conflicts ---
    print(f"\n[4] Checking Conflict Rules...")
    try:
        with open(MAP_FILE, 'r') as f:
            map_data = yaml.safe_load(f)
    except Exception as e:
        print(f"   ❌ Error reading map file: {e}")
        return

    ind_matches = {1: 0, 2: 0, 3: 0}
    sec_matches = {1: 0, 2: 0, 3: 0}

    for comm in committees:
        print(f"   ➡️  Analyzing Committee: {comm}")
        rules = map_data.get('committees').get(comm[0:4])
        
        if not rules:
            print(f"       ⚠️  No rules defined for {comm} in map file.")
            continue
            
        # Sector Check
        comm_sectors = rules.get('sectors', {})
        if sector and sector in comm_sectors:
            score = comm_sectors[sector]
            print(f"       🚨 SECTOR MATCH: '{sector}' = Level {score}")
            if score in sec_matches: sec_matches[score] = 1
        else:
             print(f"       - No sector match for '{sector}'")

        # Industry Check
        comm_industries = rules.get('industries', {})
        if industry and industry in comm_industries:
            score = comm_industries[industry]
            print(f"       🚨 INDUSTRY MATCH: '{industry}' = Level {score}")
            if score in ind_matches: ind_matches[score] = 1
        else:
             print(f"       - No industry match for '{industry}'")

    # --- Step 5: Final Output ---
    print(f"\n{'='*60}")
    print("FINAL OUTPUT FLAGS")
    print(f"{'='*60}")
    print(f"Industry match 1: {ind_matches[1]}")
    print(f"Industry match 2: {ind_matches[2]}")
    print(f"Industry match 3: {ind_matches[3]}")
    print("-" * 20)
    print(f"Sector match 1:   {sec_matches[1]}")
    print(f"Sector match 2:   {sec_matches[2]}")
    print(f"Sector match 3:   {sec_matches[3]}")
    print(f"{'='*60}\n")

if __name__ == "__main__":
    # You can run via command line: python test_script.py MSFT P000197 2020-05-20
    if len(sys.argv) == 4:
        test_transaction(sys.argv[1], sys.argv[2], sys.argv[3])
    else:
        # Or edit these values manually to test quickly
        TEST_TICKER = "A"
        TEST_BIOGUIDE = "W000800"  # Nancy Pelosi example ID (replace with real one)
        TEST_DATE = "2019-05-23"
        
        test_transaction(TEST_TICKER, TEST_BIOGUIDE, TEST_DATE)



TESTING TRANSACTION: A | W000800 | 2019-05-23

[1] Looking up Metadata for A...
   ✅ Found Industry: Diagnostics & Research
   ✅ Found Sector:   Healthcare

[2] Locating Committee History File...
   ✅ Selected File: 2019-05-03T17-13-57Z_2579cea_committee-membership.yaml
   📅 File Date:     2019-05-03 17:13:57

[3] Finding Committees for W000800 in this file...
   ✅ Member found in: ['HLIG', 'HSGO', 'HSGO06', 'HSIF', 'HSIF03', 'HSIF14', 'HSIF16', 'HLIG06', 'HLIG05']

[4] Checking Conflict Rules...
   ➡️  Analyzing Committee: HLIG
       - No sector match for 'Healthcare'
       - No industry match for 'Diagnostics & Research'
   ➡️  Analyzing Committee: HSGO
       🚨 SECTOR MATCH: 'Healthcare' = Level 2
       - No industry match for 'Diagnostics & Research'
   ➡️  Analyzing Committee: HSGO06
       🚨 SECTOR MATCH: 'Healthcare' = Level 2
       - No industry match for 'Diagnostics & Research'
   ➡️  Analyzing Committee: HSIF
       🚨 SECTOR MATCH: 'Healthcare' = Level 3
       - No ind

In [84]:
map_data

NameError: name 'map_data' is not defined

In [85]:
try:
    with open(MAP_FILE, 'r') as f:
        map_data = yaml.safe_load(f)
except Exception as e:
       print(f"   ❌ Error reading map file: {e}")
MAP_FILE = 'commette_industry_map.yaml'

rules = map_data

In [89]:
map_data.get('committees')

{'HSAS': {'name': 'House Armed Services',
  'sectors': {'Industrials': 2, 'Technology': 1},
  'industries': {'Aerospace & Defense': 3,
   'Marine Shipping': 3,
   'Uranium': 2,
   'Information Technology Services': 1,
   'Communication Equipment': 1}},
 'HSIF': {'name': 'House Energy & Commerce',
  'description': "The 'Everything' Committee - huge regulatory power",
  'sectors': {'Energy': 3,
   'Healthcare': 3,
   'Utilities': 3,
   'Communication Services': 2,
   'Technology': 2,
   'Consumer Defensive': 1},
  'industries': {'Drug Manufacturers - General': 3,
   'Biotechnology': 3,
   'Medical Devices': 3,
   'Healthcare Plans': 3,
   'Telecom Services': 3,
   'Utilities - Regulated Electric': 3,
   'Oil & Gas Integrated': 3,
   'Internet Content & Information': 2}},
 'HSWM': {'name': 'House Ways & Means',
  'description': 'Tax Policy - affects profitability directly',
  'sectors': {'Financial Services': 2,
   'Healthcare': 2,
   'Energy': 2,
   'Consumer Cyclical': 1},
  'industries

In [93]:
'ciao'[0:3]

'cia'